In [3]:
import requests
import pandas as pd
YOUR_KEY = 'I7BU1yoA12r5RaYCvba8eC'

#### Speaker scrape

In [4]:
## SPeaker scrape
from bs4 import BeautifulSoup
LINK = 'https://ic2s2-2025.org/program/'



r = requests.get(LINK)  
soup = BeautifulSoup(r.content)





# 1. Find the specific table containing the schedule
schedule_table = soup.find(id="summary")

keynote_speakers = []



# 2. Iterate through all stripped text strings within that table
# .stripped_strings is useful here because it handles the <br/> tags 
# that separate speakers in the same cell automatically.
if schedule_table:
    for text in schedule_table.stripped_strings:
        #print(text)
        if "Keynote:" in text:
            # 3. Clean the string to get just the name
            # Split by "Keynote:" and take the last part, then strip whitespace
            name = text.split("Keynote:")[-1].strip()
            keynote_speakers.append(name)

# Output the results
print("Found Keynote Speakers:")
for speaker in keynote_speakers:
    print(f"- {speaker}")

print(len(keynote_speakers))

Found Keynote Speakers:
- Dean Eckles
- Kathleen Carley
- Laura Nelson
- Duncan J. Watts
- Brandon Stewart
- Lisa P. Argyle
- Amir Goldberg
- Arnout van de Rijt
- Sarah Williams
9


#### df builder

In [5]:
# 1. Setup URL og Søgning

keynote_speakers
NAME = 'Sarah Williams'
BASE_URL = 'https://api.openalex.org/'
RESOURCE = 'authors'


# Vi kombinerer ikke navnet direkte i URL'en, men sender det som data
COMPLETE_URL = BASE_URL + RESOURCE


# 2. Definer parametre
# OpenAlex bruger 'search' parameteren til at lede i tekst
parameters = {
    'search': NAME,
    'api_key': YOUR_KEY}


seleceted_keys = ['id', 'display_name', 'works_count', 'summary_stats', 'works_api_url', 'last_known_institutions' ]

df = pd.DataFrame()
for speaker in keynote_speakers:
    parameters['search'] = speaker
    response = requests.get(COMPLETE_URL, params=parameters)
    dct = response.json()
    if dct['results']:
        results = dct['results'][0]
        subset = {}
        for key in seleceted_keys:
            if key == 'summary_stats':
                subset['h_index'] = results.get(key, {}).get('h_index')
            elif key == 'last_known_institutions':
                subset['country_code'] = results.get(key, [{}])[0].get('country_code') if results.get(key) else None
            else:
                subset[key] = results.get(key)
        df = pd.concat([df, pd.DataFrame([subset])], ignore_index=True)
df



,id,display_name,works_count,h_index,works_api_url,country_code
0,https://openalex.org/A5080265907,Dean Eckles,121,34,https://api.openalex.org/works?filter=author.i...,US
1,https://openalex.org/A5085927300,Kathleen M. Carley,881,80,https://api.openalex.org/works?filter=author.i...,None
2,https://openalex.org/A5111667138,Laura E. Nelson,21,12,https://api.openalex.org/works?filter=author.i...,US
3,https://openalex.org/A5000679279,Duncan J. Watts,233,70,https://api.openalex.org/works?filter=author.i...,US
4,https://openalex.org/A5113226689,Brandon Stewart,303,31,https://api.openalex.org/works?filter=author.i...,US
5,https://openalex.org/A5070497645,Lisa P. Argyle,38,11,https://api.openalex.org/works?filter=author.i...,None
6,https://openalex.org/A5109219790,Amir Pnueli,367,88,https://api.openalex.org/works?filter=author.i...,IL
7,https://openalex.org/A5081982677,Arnout van de Rijt,106,23,https://api.openalex.org/works?filter=author.i...,IT
8,https://openalex.org/A5033458280,Sarah Williams,123,34,https://api.openalex.org/works?filter=author.i...,GB


In [6]:
# Split ved '/' og tag det sidste element
df['ids'] = df['id'].str.split('/').str[-1]
df


,id,display_name,works_count,h_index,works_api_url,country_code,ids
0,https://openalex.org/A5080265907,Dean Eckles,121,34,https://api.openalex.org/works?filter=author.i...,US,A5080265907
1,https://openalex.org/A5085927300,Kathleen M. Carley,881,80,https://api.openalex.org/works?filter=author.i...,None,A5085927300
2,https://openalex.org/A5111667138,Laura E. Nelson,21,12,https://api.openalex.org/works?filter=author.i...,US,A5111667138
3,https://openalex.org/A5000679279,Duncan J. Watts,233,70,https://api.openalex.org/works?filter=author.i...,US,A5000679279
4,https://openalex.org/A5113226689,Brandon Stewart,303,31,https://api.openalex.org/works?filter=author.i...,US,A5113226689
5,https://openalex.org/A5070497645,Lisa P. Argyle,38,11,https://api.openalex.org/works?filter=author.i...,None,A5070497645
6,https://openalex.org/A5109219790,Amir Pnueli,367,88,https://api.openalex.org/works?filter=author.i...,IL,A5109219790
7,https://openalex.org/A5081982677,Arnout van de Rijt,106,23,https://api.openalex.org/works?filter=author.i...,IT,A5081982677
8,https://openalex.org/A5033458280,Sarah Williams,123,34,https://api.openalex.org/works?filter=author.i...,GB,A5033458280


## Snowballing

In [7]:
results = dct['results'][0]
for i, j in results.items():
    print(i,j)
    
len(dct['results'][0]['topic_share'])


id https://openalex.org/A5033458280
orcid https://orcid.org/0000-0001-5964-1329
display_name Sarah Williams
display_name_alternatives ['Alison Lye', 'B Page', 'Campain N', 'Marc Lyons', 'Mark Tuthill', 'S Williams', 'S. Williams', 'Sarah V Williams', 'Sarah V. Williams', 'Sarah Williams']
relevance_score 10620.458
works_count 123
cited_by_count 8439
summary_stats {'2yr_mean_citedness': 5.538461538461538, 'h_index': 34, 'i10_index': 54}
ids {'openalex': 'https://openalex.org/A5033458280', 'orcid': 'https://orcid.org/0000-0001-5964-1329'}
affiliations [{'institution': {'id': 'https://openalex.org/I102064193', 'ror': 'https://ror.org/01tevnk56', 'display_name': 'University of Siena', 'country_code': 'IT', 'type': 'education', 'lineage': ['https://openalex.org/I102064193']}, 'years': [2021]}, {'institution': {'id': 'https://openalex.org/I124357947', 'ror': 'https://ror.org/04cw6st05', 'display_name': 'University of London', 'country_code': 'GB', 'type': 'education', 'lineage': ['https://op

5

In [8]:
import requests

# 1. Antag at du har en liste med works_api_url'er fra dine 9 forskere
works_urls = [url for url in df['works_api_url']]
all_works = []
# 2. Loop igennem hver URL
for url in works_urls:
    # Vi tilføjer 'select' og 'mailto' som parametre til den eksisterende URL
    params = {
        'select': 'id,publication_year,cited_by_count,authorships,title,abstract_inverted_index',
        'per_page': 200,
        'api_key': YOUR_KEY
        
    }
    
    response = requests.get(url, params=params)
    
    if response.status_code == 200:
        data = response.json()
        works = data.get('results', [])
        
        for work in works:
            # Vi gemmer data i en ordbog, så det er nemt at bruge senere
            work_data = {
                'id': work.get('id'),
                'publication_year': work.get('publication_year'),
                'cited_by_count': work.get('cited_by_count'),
                'title': work.get('title'),
                'author_ids': [auth['author']['id'] for auth in work.get('authorships', [])],
                'abstract_inverted_index': work.get('abstract_inverted_index')
            }
            all_works.append(work_data)
            
            #print(f"Hentet: {work_data['title'][:50]}...")
    else:
        print(f"Fejl ved {url}: {response.status_code}")

# Nu ligger alt data for alle 9 forskere i listen 'all_works'
print(f"\nFærdig! Hentet i alt {len(all_works)} værker.")


Færdig! Hentet i alt 1198 værker.


In [9]:
all_works   

[{'id': 'https://openalex.org/W3124726575',
  'publication_year': 2021,
  'cited_by_count': 1010,
  'title': 'Shifting attention to accuracy can reduce misinformation online',
  'author_ids': ['https://openalex.org/A5020533147',
   'https://openalex.org/A5076651767',
   'https://openalex.org/A5052536455',
   'https://openalex.org/A5021501103',
   'https://openalex.org/A5080265907',
   'https://openalex.org/A5056499434'],
  'abstract_inverted_index': None},
 {'id': 'https://openalex.org/W2143331350',
  'publication_year': 2007,
  'cited_by_count': 276,
  'title': 'Over-exposed?',
  'author_ids': ['https://openalex.org/A5033859339',
   'https://openalex.org/A5080265907',
   'https://openalex.org/A5039080287',
   'https://openalex.org/A5101826175',
   'https://openalex.org/A5054248372',
   'https://openalex.org/A5086610841'],
  'abstract_inverted_index': {'As': [0],
   'sharing': [1, 132],
   'personal': [2],
   'media': [3, 22, 37, 131],
   'online': [4, 57],
   'becomes': [5],
   'easie

In [10]:
D2 = pd.DataFrame(all_works)

D3 = D2[['id', 'title', 'abstract_inverted_index']]

D2 = D2[['id', 'publication_year', 'cited_by_count', 'author_ids']]
D2.head() 
D2['n_auth'] = D2['author_ids'].apply(len)




In [11]:
mask = D2['cited_by_count'] >= 10

D2_only_good = D2[mask]
print(D2_only_good.shape, D2.shape)
mask2 = D2_only_good['n_auth'] < 10
D2_only_good =  D2_only_good[mask2]
print(D2_only_good.shape, D2.shape)





(704, 5) (1198, 5)
(652, 5) (1198, 5)


In [12]:
(works_urls)

['https://api.openalex.org/works?filter=author.id:A5080265907',
 'https://api.openalex.org/works?filter=author.id:A5085927300',
 'https://api.openalex.org/works?filter=author.id:A5111667138',
 'https://api.openalex.org/works?filter=author.id:A5000679279',
 'https://api.openalex.org/works?filter=author.id:A5113226689',
 'https://api.openalex.org/works?filter=author.id:A5070497645',
 'https://api.openalex.org/works?filter=author.id:A5109219790',
 'https://api.openalex.org/works?filter=author.id:A5081982677',
 'https://api.openalex.org/works?filter=author.id:A5033458280']

In [14]:
import requests
import pandas as pd
import time

# --- KONFIGURATION ---
WORKING_URL = 'https://api.openalex.org/works'
EMAIL = "s245109@dtu.dk"  # Din email til Polite Pool
API_KEY = YOUR_KEY

# 1. FORBEREDELSE AF DATA
# Vi antager at 'df' findes. Hvis du tester uden 'df', så lav en dummy her:
# df = pd.DataFrame({'id': ['https://openalex.org/A5080265907'], 'works_count': [100]})

# Filtrer forfattere (5 til 5000 værker)
df_filtered_authors = df[(df['works_count'] >= 5) & (df['works_count'] <= 5000)].copy()

# RENS ID'er: Vi skal fjerne "https://openalex.org/" og kun beholde "A..."
# Dette er vigtigt, ellers virker filteret ikke

ids = df_filtered_authors['id'].tolist()
idstr = '|'.join(ids) # Samler dem med "ELLER" (OR)

print(f"Søger efter værker for {len(ids)} forfattere...")

# 2. DEFINER KONCEPTER (CSS FILTER)
social_ids = "C144024400|C15744967|C162324750|C17744445"
quant_ids = "C33923547|C121332964|C41008148"

# 3. OPSÆTNING AF API PARAMETRE
# Vi starter cursoren ved '*'
current_cursor = '*'
all_works = []
page_num = 1

# Byg filter-strengen én gang
# Logik: (Forfattere) AND (Få forfattere) AND (Social) AND (Quant) AND (Citationer)
filter_string = (
    f'author.id:{idstr},'
    'authors_count:<10,'
    f'concepts.id:{social_ids},'
    f'concepts.id:{quant_ids},'
    'cited_by_count:>10'
)

print("Starter API loop...")

# 4. INDSAMLING (WHILE LOOP)
while True:
    params = {
        'filter': filter_string,
        'select': 'id,publication_year,cited_by_count,authorships,title,abstract_inverted_index,concepts',
        'per_page': 200,
        'cursor': current_cursor,
        'mailto': EMAIL 
    }
    
    # Tilføj API key hvis den findes
    if API_KEY:
        params['api_key'] = API_KEY

    try:
        response = requests.get(WORKING_URL, params=params)
        
        if response.status_code == 200:
            data = response.json()
            results = data.get('results', [])
            meta = data.get('meta', {})
            
            # Hvis listen er tom, er vi færdige
            if not results:
                print("Ingen flere resultater fundet.")
                break
                
            # Gem resultater
            for work in results:
                all_works.append({
                    'id': work.get('id'),
                    'publication_year': work.get('publication_year'),
                    'cited_by_count': work.get('cited_by_count'),
                    'title': work.get('title'),
                    'author_ids': [auth['author']['id'] for auth in work.get('authorships', [])],
                    'abstract_inverted_index': work.get('abstract_inverted_index')
                    # Vi behøver ikke gemme concepts længere, da vi har filtreret på dem
                })
            
            print(f"Side {page_num} hentet ({len(results)} værker). Total: {len(all_works)}")
            
            # OPDATER CURSOR TIL NÆSTE RUNDE
            current_cursor = meta.get('next_cursor')
            page_num += 1
            
            # Sikkerhedsnet: Hvis next_cursor mangler, stop
            if not current_cursor:
                break
                
        elif response.status_code == 429:
            print("For mange forespørgsler (429). Venter 1 sekund...")
            time.sleep(1)
            continue # Prøv igen uden at ændre cursor
            
        else:
            print(f"Fejl opstod: {response.status_code} - {response.text}")
            break

    except Exception as e:
        print(f"Kritisk fejl: {e}")
        break

# 5. RESULTAT BEHANDLING
print(f"\nFærdig! Hentet i alt {len(all_works)} værker.")

if all_works:
    D2 = pd.DataFrame(all_works)
    
    # Vis de første rækker
    display(D2.head())
    
    # Vælg kolonner
    final_columns = ['id', 'publication_year', 'cited_by_count', 'author_ids', 'title', 'abstract_inverted_index']
    
    # Her sikrer vi os, at kolonnerne findes før vi vælger dem
    existing_cols = [c for c in final_columns if c in D2.columns]
    D2_final = D2[existing_cols]
    
    print(f"Endeligt dataset størrelse: {len(D2_final)}")
else:
    print("Ingen data fundet der matcher dine kriterier.")

Søger efter værker for 9 forfattere...
Starter API loop...
Side 1 hentet (200 værker). Total: 200
Side 2 hentet (175 værker). Total: 375
Ingen flere resultater fundet.

Færdig! Hentet i alt 375 værker.


,id,publication_year,cited_by_count,title,author_ids,abstract_inverted_index
0,https://openalex.org/W2095655043,2013,2999,Text as Data: The Promise and Pitfalls of Auto...,"[https://openalex.org/A5090665793, https://ope...","{'Politics': [0], 'and': [1, 9, 67, 96, 99, 10..."
1,https://openalex.org/W2147453867,2006,2124,Experimental Study of Inequality and Unpredict...,"[https://openalex.org/A5058815871, https://ope...","{'Hit': [0], 'songs,': [1], 'books,': [2], 'an..."
2,https://openalex.org/W2150375325,2007,1865,"Influentials, Networks, and Public Opinion For...","[https://openalex.org/A5000679279, https://ope...","{'A': [0], 'central': [1], 'idea': [2], 'in': ..."
3,https://openalex.org/W2096974619,2014,1821,Structural Topic Models for Open‐Ended Survey ...,"[https://openalex.org/A5082742221, https://ope...","{'Collection': [0], 'and': [1, 14, 37, 78, 97,..."
4,https://openalex.org/W2049607688,2006,1695,Empirical Analysis of an Evolving Social Network,"[https://openalex.org/A5066696247, https://ope...","{'Social': [0], 'networks': [1], 'evolve': [2]..."


Endeligt dataset størrelse: 375


In [19]:
# ... (Din forberedelse af df_filtered_authors og koncepter foroven er uændret) ...

# 1. RENS OG FORBERED ID-LISTE
# Sørg for at 'ids' kun indeholder ID'er (fx "A5080265907") og ikke hele URL'er
ids = df_filtered_authors['id']#.astype(str).str.split('/').str[-1].tolist()

BATCH_SIZE = 25
all_works = [] # Denne skal ligge UDENFOR loops, så vi samler alt op

print(f"Starter indsamling for {len(ids)} forfattere i batches af {BATCH_SIZE}...")

# --- YDRE LOOP: BATCHING ---
# Vi bruger range() til at hoppe med 25 ad gangen (0, 25, 50...)
for i in range(0, len(ids), BATCH_SIZE):
    
    # Udvælg de næste 25 ID'er
    batch_ids = ids[i : i + BATCH_SIZE]
    batch_idstr = '|'.join(batch_ids)
    
    print(f"--- Behandler batch {i//BATCH_SIZE + 1} (Forfatter {i+1} til {min(i+BATCH_SIZE, len(ids))}) ---")

    # VIGTIGT: Cursor skal nulstilles for hver ny batch!
    current_cursor = '*'
    page_num = 1
    
    # Byg filter-strengen for DENNE batch
    filter_string = (
        f'author.id:{batch_idstr},'
        'authors_count:<10,'
        f'concepts.id:{social_ids},'
        f'concepts.id:{quant_ids},'
        'cited_by_count:>10'
    )

    # --- INDRE LOOP: API PAGINATION (Din eksisterende logik) ---
    while True:
        params = {
            'filter': filter_string,
            'select': 'id,publication_year,cited_by_count,authorships,title,abstract_inverted_index,concepts',
            'per_page': 200,
            'cursor': current_cursor,
            'mailto': EMAIL 
        }
        
        if API_KEY:
            params['api_key'] = API_KEY

        try:
            response = requests.get(WORKING_URL, params=params)
            
            if response.status_code == 200:
                data = response.json()
                results = data.get('results', [])
                meta = data.get('meta', {})
                
                # Hvis listen er tom, er denne batch færdig -> Break indre loop
                if not results:
                    print(f"   Batch færdig. Ingen flere sider.")
                    break
                    
                # Gem resultater
                for work in results:
                    all_works.append({
                        'id': work.get('id'),
                        'publication_year': work.get('publication_year'),
                        'cited_by_count': work.get('cited_by_count'),
                        'title': work.get('title'),
                        'author_ids': [auth['author']['id'] for auth in work.get('authorships', [])],
                        'abstract_inverted_index': work.get('abstract_inverted_index')
                    })
                
                print(f"   Side {page_num} hentet ({len(results)} værker). Total indsamlet: {len(all_works)}")
                
                # Opdater cursor
                current_cursor = meta.get('next_cursor')
                page_num += 1
                
                if not current_cursor:
                    break
                    
            elif response.status_code == 429:
                print("   For mange forespørgsler (429). Venter 1 sekund...")
                time.sleep(1)
                continue 
                
            else:
                print(f"   Fejl opstod: {response.status_code} - {response.text}")
                break # Hopper ud af while-loopet (til næste batch)

        except Exception as e:
            print(f"   Kritisk fejl: {e}")
            break

# 5. RESULTAT BEHANDLING (Uændret)
print(f"\nHelt færdig! Hentet i alt {len(all_works)} værker.")
# ... Resten af din DataFrame kode ...

Starter indsamling for 9 forfattere i batches af 25...
--- Behandler batch 1 (Forfatter 1 til 9) ---
   Side 1 hentet (200 værker). Total indsamlet: 200
   Side 2 hentet (175 værker). Total indsamlet: 375
   Batch færdig. Ingen flere sider.

Helt færdig! Hentet i alt 375 værker.
